In [1]:
# ============================================================
# CONFIGURATION
# ============================================================
# Import des librairies necessaires
# Definition des chemins des dossiers
# ============================================================

import sqlite3    # librairie Python standard pour SQLite
import pandas as pd  # manipulation de tableaux de donnees

# Chemins des dossiers de donnees
PROCESSED = "../data/processed/"  # dossier des donnees nettoyees

print("Configuration OK !")

Configuration OK !


In [2]:
# ============================================================
# CREATION BASE DE DONNEES SQL
# ============================================================
# On cree une base de donnees SQLite structuree en 4 tables
# selon notre Modele Conceptuel de Donnees (MCD).
#
# CHOIX DE SQLITE :
# SQLite est la solution la plus adaptee a un POC car :
# - Aucune installation requise
# - Produit un seul fichier portable .db
# - Librairie sqlite3 nativement integree a Python
# - Parfaitement adapte au volume de 2440 lignes
# Pour une mise en production on migrerait vers PostgreSQL.
#
# STRUCTURE EN 4 TABLES :
# Cette structure respecte notre MCD et evite la redondance
# des donnees grace aux cles etrangeres (FOREIGN KEY).
#
# TABLE 1 — groupe_politique
# Table de reference contenant les 5 groupes politiques
# avec leurs candidats pour 2017 et 2022.
#
# TABLE 2 — commune
# Contient les donnees geographiques fixes de chaque commune.
# Une ligne par commune car la superficie et la densite
# ne changent pas dans le temps.
#
# TABLE 3 — indicateurs
# Contient les donnees socio-economiques par commune et annee.
# Deux lignes par commune : une pour 2016 et une pour 2021.
# On utilise les annees des indicateurs (2016/2021) et non
# les annees electorales (2017/2022) car ce sont les annees
# reelles des donnees INSEE.
# nb_associations_p_mille est une donnee statique (vie
# associative) appliquee de la meme facon pour 2016 et 2021.
#
# TABLE 4 — resultat_electoral
# Contient les scores et le groupe gagnant par commune
# et par annee electorale (2017 et 2022).
# La cle etrangere id_groupe fait reference a groupe_politique
# pour respecter l integrite referentielle.
#
# POURQUOI PAS NOSQL :
# Nos donnees sont structurees et tabulaires avec des relations
# claires entre les entites. SQL est plus adapte que NoSQL
# pour les analyses statistiques et les jointures.
# ============================================================

# Charger le dataset final
df = pd.read_csv(
    PROCESSED + "dataset_bretagne.csv",
    dtype={
        'code_commune' : str,
        'code_dept'    : str,
        'code_insee'   : str
    }
)
print("Dataset charge :", df.shape)

# Connexion a la base SQLite
# Le fichier .db est cree automatiquement s il n existe pas
conn   = sqlite3.connect(PROCESSED + "electio_analytics.db")
cursor = conn.cursor()

# Activer la verification des cles etrangeres
cursor.execute("PRAGMA foreign_keys = ON")

# Supprimer les tables si elles existent deja
# pour permettre de relancer le script proprement
cursor.execute("DROP TABLE IF EXISTS resultat_electoral")
cursor.execute("DROP TABLE IF EXISTS indicateurs")
cursor.execute("DROP TABLE IF EXISTS commune")
cursor.execute("DROP TABLE IF EXISTS groupe_politique")

# ============================================================
# TABLE 1 — groupe_politique
# ============================================================
# Table de reference des 5 groupes politiques.
# id_groupe : cle primaire unique par groupe
# nom_groupe : nom du groupe (ext_gauche, gauche, centre...)
# candidats_2017 : liste des candidats 2017 dans ce groupe
# candidats_2022 : liste des candidats 2022 dans ce groupe
# ============================================================

cursor.execute("""
    CREATE TABLE groupe_politique (
        id_groupe       INTEGER PRIMARY KEY,
        nom_groupe      TEXT    NOT NULL,
        candidats_2017  TEXT,
        candidats_2022  TEXT
    )
""")

groupes = [
    (1, 'ext_gauche',
        'Arthaud, Poutou',
        'Arthaud, Poutou'),
    (2, 'gauche',
        'Melenchon, Hamon, Jadot',
        'Melenchon, Hidalgo, Jadot, Roussel'),
    (3, 'centre',
        'Macron',
        'Macron'),
    (4, 'droite',
        'Fillon',
        'Pecresse'),
    (5, 'ext_droite',
        'Le Pen, Dupont-Aignan, Lassalle, Asselineau, Cheminade',
        'Le Pen, Zemmour, Dupont-Aignan, Lassalle'),
]

cursor.executemany(
    "INSERT INTO groupe_politique VALUES (?, ?, ?, ?)",
    groupes
)
print("Table groupe_politique creee :", len(groupes), "lignes")

# ============================================================
# TABLE 2 — commune
# ============================================================
# Donnees geographiques fixes par commune.
# code_insee : cle primaire sur 5 caracteres (ex : 22001)
# superficie_km2 : superficie en km2
# densite_hab_km2 : densite de population par km2
# Ces donnees sont fixes dans le temps donc une seule
# ligne par commune independamment de l annee.
# ============================================================

cursor.execute("""
    CREATE TABLE commune (
        code_insee      TEXT PRIMARY KEY,
        code_commune    TEXT NOT NULL,
        nom_commune     TEXT NOT NULL,
        code_dept       TEXT NOT NULL,
        superficie_km2  REAL,
        densite_hab_km2 REAL
    )
""")

# Une ligne par commune unique
communes_unique = df.drop_duplicates('code_insee')[[
    'code_insee', 'code_commune', 'nom_commune',
    'code_dept', 'superficie_km2', 'densite'
]].copy()
communes_unique.columns = [
    'code_insee', 'code_commune', 'nom_commune',
    'code_dept', 'superficie_km2', 'densite_hab_km2'
]

cursor.executemany(
    "INSERT INTO commune VALUES (?, ?, ?, ?, ?, ?)",
    communes_unique.values.tolist()
)
print("Table commune creee :", len(communes_unique), "lignes")

# ============================================================
# TABLE 3 — indicateurs
# ============================================================
# Donnees socio-economiques par commune et par annee.
# id_indicateur : cle primaire auto-incrementee
# code_insee : cle etrangere vers la table commune
# annee : 2016 (proxy pour election 2017)
#         ou 2021 (proxy pour election 2022)
# taux_chomage : chomeurs / actifs * 100
# revenu_median : revenu median en euros
# taux_criminalite : infractions pour 1000 habitants
# nb_entreprises_p_mille : entreprises pour 1000 habitants
# nb_associations_p_mille : associations actives pour 1000 habitants
#                            (donnee statique, identique 2016/2021)
# pct_sans_diplome : % personnes sans diplome
# pct_diplome_sup : % personnes diplome superieur
# ============================================================

cursor.execute("""
    CREATE TABLE indicateurs (
        id_indicateur           INTEGER PRIMARY KEY AUTOINCREMENT,
        code_insee              TEXT    NOT NULL,
        annee                   INTEGER NOT NULL,
        population              REAL,
        taux_chomage            REAL,
        revenu_median           REAL,
        taux_criminalite        REAL,
        nb_entreprises_p_mille  REAL,
        nb_associations_p_mille REAL,
        pct_sans_diplome        REAL,
        pct_diplome_sup         REAL,
        FOREIGN KEY (code_insee) REFERENCES commune(code_insee)
    )
""")

# Indicateurs 2016 depuis les lignes annee=2017
ind_2016 = df[df['annee'] == 2017][[
    'code_insee', 'population',
    'taux_chomage_2016', 'revenu_median_2016',
    'taux_criminalite_2016', 'nb_entreprises_pour_mille_2016',
    'nb_associations_pour_mille',
    'pct_sans_diplome_2016', 'pct_diplome_sup_2016'
]].copy()
ind_2016.insert(1, 'annee', 2016)
ind_2016.columns = [
    'code_insee', 'annee', 'population', 'taux_chomage',
    'revenu_median', 'taux_criminalite', 'nb_entreprises_p_mille',
    'nb_associations_p_mille',
    'pct_sans_diplome', 'pct_diplome_sup'
]

# Indicateurs 2021 depuis les lignes annee=2022
ind_2021 = df[df['annee'] == 2022][[
    'code_insee', 'population',
    'taux_chomage_2021', 'revenu_median_2021',
    'taux_criminalite_2021', 'nb_entreprises_pour_mille_2021',
    'nb_associations_pour_mille',
    'pct_sans_diplome_2021', 'pct_diplome_sup_2021'
]].copy()
ind_2021.insert(1, 'annee', 2021)
ind_2021.columns = [
    'code_insee', 'annee', 'population', 'taux_chomage',
    'revenu_median', 'taux_criminalite', 'nb_entreprises_p_mille',
    'nb_associations_p_mille',
    'pct_sans_diplome', 'pct_diplome_sup'
]

# Fusion et insertion
ind_all = pd.concat([ind_2016, ind_2021], axis=0).reset_index(drop=True)
cursor.executemany(
    """INSERT INTO indicateurs
       (code_insee, annee, population, taux_chomage, revenu_median,
        taux_criminalite, nb_entreprises_p_mille, nb_associations_p_mille,
        pct_sans_diplome, pct_diplome_sup)
       VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
    ind_all.values.tolist()
)
print("Table indicateurs creee :", len(ind_all), "lignes")

# ============================================================
# TABLE 4 — resultat_electoral
# ============================================================
# Scores electoraux et groupe gagnant par commune et annee.
# id_resultat : cle primaire auto-incrementee
# code_insee : cle etrangere vers commune
# annee : 2017 ou 2022 (annees electorales)
# score_X : pourcentage de voix du groupe X
# id_groupe : cle etrangere vers groupe_politique
#             identifie le groupe gagnant de la commune
# ============================================================

cursor.execute("""
    CREATE TABLE resultat_electoral (
        id_resultat      INTEGER PRIMARY KEY AUTOINCREMENT,
        code_insee       TEXT    NOT NULL,
        annee            INTEGER NOT NULL,
        nb_inscrits      INTEGER,
        nb_votants       INTEGER,
        taux_abstention  REAL,
        score_ext_gauche REAL,
        score_gauche     REAL,
        score_centre     REAL,
        score_droite     REAL,
        score_ext_droite REAL,
        id_groupe        INTEGER,
        FOREIGN KEY (code_insee) REFERENCES commune(code_insee),
        FOREIGN KEY (id_groupe)  REFERENCES groupe_politique(id_groupe)
    )
""")

# Mapper le nom du gagnant vers son id_groupe
mapping_groupe = {
    'ext_gauche' : 1,
    'gauche'     : 2,
    'centre'     : 3,
    'droite'     : 4,
    'ext_droite' : 5,
}

resultats = df[[
    'code_insee', 'annee', 'nb_inscrits', 'nb_votants',
    'taux_abstention', 'score_ext_gauche', 'score_gauche',
    'score_centre', 'score_droite', 'score_ext_droite', 'gagnant'
]].copy()
resultats['id_groupe'] = resultats['gagnant'].map(mapping_groupe)
resultats = resultats.drop(columns=['gagnant'])

cursor.executemany(
    """INSERT INTO resultat_electoral
       (code_insee, annee, nb_inscrits, nb_votants, taux_abstention,
        score_ext_gauche, score_gauche, score_centre, score_droite,
        score_ext_droite, id_groupe)
       VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
    resultats.values.tolist()
)
print("Table resultat_electoral creee :", len(resultats), "lignes")

# Sauvegarder et fermer la connexion
conn.commit()
conn.close()
print()
print("Base SQLite sauvegardee : electio_analytics.db")
print()

# ============================================================
# VERIFICATION FINALE
# ============================================================
# On relit la base pour verifier que tout est bien sauvegarde
# ============================================================

conn = sqlite3.connect(PROCESSED + "electio_analytics.db")
print("Verification des tables :")
for table in ['groupe_politique', 'commune', 'indicateurs', 'resultat_electoral']:
    nb = pd.read_sql(f"SELECT COUNT(*) as nb FROM {table}", conn).iloc[0, 0]
    print(f"  {table:25s} : {nb} lignes")

print()
print("Exemple requete SQL — Top 5 communes avec plus de chomage en 2016 :")
query = """
    SELECT c.nom_commune, c.code_dept, i.taux_chomage
    FROM indicateurs i
    JOIN commune c ON i.code_insee = c.code_insee
    WHERE i.annee = 2016
    ORDER BY i.taux_chomage DESC
    LIMIT 5
"""
print(pd.read_sql(query, conn))
conn.close()

Dataset charge : (2440, 30)
Table groupe_politique creee : 5 lignes
Table commune creee : 1233 lignes
Table indicateurs creee : 2440 lignes
Table resultat_electoral creee : 2440 lignes

Base SQLite sauvegardee : electio_analytics.db

Verification des tables :
  groupe_politique          : 5 lignes
  commune                   : 1233 lignes
  indicateurs               : 2440 lignes
  resultat_electoral        : 2440 lignes

Exemple requete SQL — Top 5 communes avec plus de chomage en 2016 :
              nom_commune code_dept  taux_chomage
0                Guingamp        22     24.675037
1               Langoëlan        56     23.648649
2        Peumerit-Quintin        22     22.955778
3              Loqueffret        29     21.491067
4  Saint-Caradec-Trégomel        56     21.004789
